# **Task 1: Parquet Internals and Metadata Inspection*
**Step 1: Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use np.random.seed(42) for reproducibility.**

In [3]:
pip install pandas numpy pyarrow duckdb 

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime,timedelta 
import random

In [5]:
np.random.seed(42)

In [6]:
n = 500_000

In [7]:
today = datetime.now()
today

datetime.datetime(2026, 3, 14, 16, 0, 19, 927578)

In [8]:
three_years_ago = today - timedelta(days=3*365)
three_years_ago

datetime.datetime(2023, 3, 15, 16, 0, 19, 927578)

In [9]:
all_days = pd.date_range(start=three_years_ago, end=today, freq='D')

In [10]:
df = pd.DataFrame({
    "user_id" : np.arange(1,n+1),
    "city" : np.random.choice(["Kyoto", "Berlin", "Seoul", "Prague", "Nairobi", "Perth", "Lima", "Lisbon", "Chiang Mai", "Tbilisi"],n),
    "score" : np.random.uniform(0,100,n),
    "active" : np.random.choice([True,False],n),
    "signup_date" : np.random.choice(all_days, n),
    "age" : np.random.randint(18,81,n),
    "sessions" : np.random.randint(0,501,n),
    "revenue" :  np.random.uniform(0,1001,n)
})

In [11]:
df.shape

(500000, 8)

In [12]:
df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Lima,43.689490,False,2023-04-16 16:00:19.927578,61,401,379.897571
1,2,Prague,97.403462,True,2023-12-16 16:00:19.927578,78,475,558.140786
2,3,Lisbon,66.148238,False,2023-07-10 16:00:19.927578,28,340,65.702970
3,4,Nairobi,21.993544,False,2023-12-27 16:00:19.927578,75,142,691.242539
4,5,Lima,91.719953,True,2026-02-15 16:00:19.927578,65,431,942.428408


**Step 2: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:**

- **The number of row groups**
- **The number of rows and columns**
- **The schema (column names and types)**
- **For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)**

In [13]:
import pyarrow.parquet as pq

# 1. Save the data as Parquet format 
file_name = "users_500k.parquet"
df.to_parquet(file_name,engine='pyarrow',index=False)

# Note on Parquet Engine:
# Why should we use engine="pyarrow" to ensure the dataset is processed using the Apache Arrow framework. This guarantees:

# - Interoperability: Seamless data exchange between Python, C++, and Java.

# - Performance: Faster vectorized read/write operations for large datasets (500k rows).

# - Metadata Access: Reliable inspection of row groups and column statistics using pyarrow.parquet.

In [14]:
# 2. Open the file with Parquet file object
parquet_file = pq.ParquetFile(file_name)

In [15]:
# 3. The number of row groups
   # The number of rows and columns
   # The schema (column names and types)

print(f"--- General Information ---")
print(f"Row Groups : {parquet_file.num_row_groups}")
print(f"Row count: {parquet_file.metadata.num_rows}")
print(f"Column Count : {parquet_file.metadata.num_columns}")
print(f"\n--- Schema ---")
print(parquet_file.schema)

--- General Information ---
Row Groups : 1
Row count: 500000
Column Count : 8

--- Schema ---
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}



In [16]:
# For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

first_rg = parquet_file.metadata.row_group(0)

for i in range(first_rg.num_columns):
    stats = first_rg.column(i).statistics 

    print(f"\nColumn : {first_rg.column(i).path_in_schema}")
    print(f" - Physical type : {first_rg.column(i).physical_type}")
    print(f" - Compressed size : {first_rg.column(i).total_compressed_size} bytes")

    if stats: 
        print(f"  - Min value: {stats.min}")
        print(f"  - Max value: {stats.max}")
        print(f"  - Null count: {stats.null_count}")
    else:
        print(" - Statistic doesn't exist")


Column : user_id
 - Physical type : INT64
 - Compressed size : 2280081 bytes
  - Min value: 1
  - Max value: 500000
  - Null count: 0

Column : city
 - Physical type : BYTE_ARRAY
 - Compressed size : 251175 bytes
  - Min value: Berlin
  - Max value: Tbilisi
  - Null count: 0

Column : score
 - Physical type : DOUBLE
 - Compressed size : 4279334 bytes
  - Min value: 5.188445665327279e-05
  - Max value: 99.99983148609545
  - Null count: 0

Column : active
 - Physical type : BOOLEAN
 - Compressed size : 62553 bytes
  - Min value: False
  - Max value: True
  - Null count: 0

Column : signup_date
 - Physical type : INT64
 - Compressed size : 697308 bytes
  - Min value: 2023-03-15 16:00:19.927578
  - Max value: 2026-03-14 16:00:19.927578
  - Null count: 0

Column : age
 - Physical type : INT32
 - Compressed size : 376346 bytes
  - Min value: 18
  - Max value: 80
  - Null count: 0

Column : sessions
 - Physical type : INT32
 - Compressed size : 565610 bytes
  - Min value: 0
  - Max value: 50

**Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.**

In [17]:
import os 

# Save the same dataset as CSV file
csv_file = "users_500k.csv"
df.to_csv(csv_file,index=False)

# File sizes as byte 
parquet_size = os.path.getsize("users_500k.parquet")
csv_size = os.path.getsize(csv_file)

# Convert the size into KB
parquet_kb = parquet_size / 1024
csv_kb = csv_size / 1024

compression_ratio = csv_size / parquet_size

print(f"Parquet file size: {parquet_kb:.2f} KB")
print(f"CSV file size: {csv_kb:.2f} KB")
print(f"Compression ratio: {compression_ratio:.2f}x")
print(f"Comparison Result: Parquet is {compression_ratio:.1f}x smaller than CSV!")

Parquet file size: 12496.10 KB
CSV file size: 44190.40 KB
Compression ratio: 3.54x
Comparison Result: Parquet is 3.5x smaller than CSV!


**Step 4: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.**

**What Information Does Parquet Metadata Provide?**

Unlike a CSV file, which is a plain-text format containing only raw data without any structural information, a Parquet file stores extensive metadata in its footer. This metadata includes:

- Schema Information: Exact data types for every column (e.g., INT64, BOOLEAN, TIMESTAMP), ensuring data integrity without needing to "infer" types.
  
- Physical Structure: The number of Row Groups and the byte-offset of each column chunk.
  
- Column Statistics: Per-column statistics for every Row Group, such as:
  
       - Min / Max values
         
       - Null count
         
       - Total uncompressed and compressed size
  
      - Dictionary Encoding: Information about repeated values, which helps in efficient data representation.

**Why Does This Matter for Query Performance?**

The presence of metadata enables two critical performance optimizations that CSV cannot perform:

1. Predicate Pushdown (Row Group Skipping)
If you run a query like SELECT * FROM table WHERE age > 80, Parquet looks at the Min/Max statistics in the metadata of each Row Group before reading the actual data. If a Row Group's max_age is 75, Parquet skips that entire block of data. In a CSV, the engine would be forced to scan every single row to check the condition.

2. Column Projection (Schema Selection)
Since the metadata stores the exact location (offsets) of each column, the engine can jump directly to the specific column requested (e.g., SELECT score). In a CSV, even if you only need one column, the engine must read and parse the entire line (all columns) to find the commas separating the values.

3. Data Type Consistency
Because the metadata explicitly defines types, there is no overhead for "Type Inference." In CSVs, a tool like Pandas must scan a portion of the file to guess if a column is a float or a string, which is both slow and prone to errors.

------------------------------
# **Task 2: Column Projection and Selective Reads*

**Step 1: Read the full Parquet file from Task 1 and time it.**

In [20]:
pd.read_parquet("users_500k.parquet")

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Lima,43.689490,False,2023-04-16 16:00:19.927578,61,401,379.897571
1,2,Prague,97.403462,True,2023-12-16 16:00:19.927578,78,475,558.140786
2,3,Lisbon,66.148238,False,2023-07-10 16:00:19.927578,28,340,65.702970
3,4,Nairobi,21.993544,False,2023-12-27 16:00:19.927578,75,142,691.242539
4,5,Lima,91.719953,True,2026-02-15 16:00:19.927578,65,431,942.428408
...,...,...,...,...,...,...,...,...
499995,499996,Prague,61.531318,False,2026-02-23 16:00:19.927578,64,467,308.857911
499996,499997,Kyoto,74.712924,False,2025-04-29 16:00:19.927578,75,295,368.127030
499997,499998,Chiang Mai,48.430697,True,2025-08-31 16:00:19.927578,80,29,507.137560
499998,499999,Lisbon,89.680757,True,2025-05-04 16:00:19.927578,79,85,585.612388


In [19]:
%timeit pd.read_parquet("users_500k.parquet")

155 ms ± 3.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Step 2: Read only 2 columns from the same Parquet file and time it.**

In [22]:
pd.read_parquet("users_500k.parquet",columns=['user_id','score'])

,user_id,score
0,1,43.689490
1,2,97.403462
2,3,66.148238
3,4,21.993544
4,5,91.719953
...,...,...
499995,499996,61.531318
499996,499997,74.712924
499997,499998,48.430697
499998,499999,89.680757


**Step 3: Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).**

In [31]:
pd.read_csv("users_500k.csv")[['user_id','city']]

,user_id,city
0,1,Lima
1,2,Prague
2,3,Lisbon
3,4,Nairobi
4,5,Lima
...,...,...
499995,499996,Prague
499996,499997,Kyoto
499997,499998,Chiang Mai
499998,499999,Lisbon


In [32]:
%timeit pd.read_csv("users_500k.csv")[["user_id", "score"]]

1.15 s ± 18.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.**

**Why Parquet Selective Reads are Faster**

The performance gap between Parquet and CSV during selective reads is due to the fundamental difference between Row-Oriented and Columnar storage:

1. **Column-Chunk Layout:** As observed in Task 1, Parquet stores data in "Column Chunks" within Row Groups. Each column's data is physically stored together on disk. In contrast, CSV is row-oriented, meaning all values for a single row (user_id, city, score, etc.) are stored side-by-side.

2. **Metadata-Driven Skipping:** Because the Parquet footer contains the exact starting position (offset) and size of every column chunk, the execution engine (like PyArrow) can "jump" directly to the columns you requested. It physically ignores the bytes belonging to the other 6 columns.

3. **Reduced I/O and CPU: * CSV:** To find the "score" for the first row, the engine must read "user_id," "city," and every character/comma in between to know where "score" starts. This wastes Disk I/O and CPU cycles parsing unnecessary text.

- **Parquet:** The engine performs a "seek" operation to the exact byte where the "user_id" chunk begins and only pulls that data into memory.

--------------------------
# **Task 3: Predicate Pushdown in Practice*

**Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.**

In [34]:
pd.read_parquet("users_500k.parquet", filters=[('age', '>', 50)])

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Lima,43.689490,False,2023-04-16 16:00:19.927578,61,401,379.897571
1,2,Prague,97.403462,True,2023-12-16 16:00:19.927578,78,475,558.140786
2,4,Nairobi,21.993544,False,2023-12-27 16:00:19.927578,75,142,691.242539
3,5,Lima,91.719953,True,2026-02-15 16:00:19.927578,65,431,942.428408
4,7,Seoul,41.789961,False,2024-06-03 16:00:19.927578,65,441,646.145348
...,...,...,...,...,...,...,...,...
237856,499996,Prague,61.531318,False,2026-02-23 16:00:19.927578,64,467,308.857911
237857,499997,Kyoto,74.712924,False,2025-04-29 16:00:19.927578,75,295,368.127030
237858,499998,Chiang Mai,48.430697,True,2025-08-31 16:00:19.927578,80,29,507.137560
237859,499999,Lisbon,89.680757,True,2025-05-04 16:00:19.927578,79,85,585.612388


In [35]:
# Timing the read with Predicate Pushdown (age > 50)
%timeit pd.read_parquet("users_500k.parquet", filters=[('age', '>', 50)])

125 ms ± 7.73 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.**

In [50]:
# Step 2: Load the full file first, then filter in memory using pandas
df_full = pd.read_parquet("users_500k.parquet")
print(df_full)
print('='*50)
print(f"============= DF_FILTERED: =============")
df_filtered = df_full[df_full['age'] > 50]
print(df_filtered)

        user_id        city      score  active                signup_date  \
0             1        Lima  43.689490   False 2023-04-16 16:00:19.927578   
1             2      Prague  97.403462    True 2023-12-16 16:00:19.927578   
2             3      Lisbon  66.148238   False 2023-07-10 16:00:19.927578   
3             4     Nairobi  21.993544   False 2023-12-27 16:00:19.927578   
4             5        Lima  91.719953    True 2026-02-15 16:00:19.927578   
...         ...         ...        ...     ...                        ...   
499995   499996      Prague  61.531318   False 2026-02-23 16:00:19.927578   
499996   499997       Kyoto  74.712924   False 2025-04-29 16:00:19.927578   
499997   499998  Chiang Mai  48.430697    True 2025-08-31 16:00:19.927578   
499998   499999      Lisbon  89.680757    True 2025-05-04 16:00:19.927578   
499999   500000      Lisbon  39.645674    True 2023-07-14 16:00:19.927578   

        age  sessions     revenue  
0        61       401  379.897571  
1  

In [52]:
%timeit df_full

43.3 ns ± 1.39 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [54]:
%timeit df_filtered

41.1 ns ± 1.45 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


**Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.**

In [55]:
# 1. Filter using Predicate Pushdown
df_pushdown = pd.read_parquet("users_500k.parquet", filters=[('age', '>', 50)])

# 2. Filter using In-Memory Pandas logic
df_full = pd.read_parquet("users_500k.parquet")
df_in_memory = df_full[df_full['age'] > 50]

# Compare Results
print(f"Pushdown rows: {len(df_pushdown)}")
print(f"In-memory rows: {len(df_in_memory)}")
print(f"Results Match: {len(df_pushdown) == len(df_in_memory)}")

Pushdown rows: 237861
In-memory rows: 237861
Results Match: True


##### **1. Verification (The "Trust but Verify" Rule)**

When you use advanced optimizations like Predicate Pushdown, you are letting the "storage engine" (PyArrow) handle the logic instead of Python. We ran both methods to prove that the optimization didn't "accidentally" lose any data. Since both returned exactly 237,861 rows, you've confirmed that the faster method is 100% reliable.

##### **2. Demonstrating the Efficiency Gap**

While the output (the number of rows) is identical, the process was completely different:

- **In-Memory:** You loaded all 500,000 rows into your computer's RAM first, then asked the CPU to throw away the ones you didn't want.

- **Pushdown:** The computer looked at the Parquet metadata, saw which "groups" of data didn't have anyone over 50, and never even read them from the disk.

**Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:**

##### **Time it and compare with both PyArrow and pandas approaches.**

In [57]:
import duckdb

# Use the actual filename you created in Task 1
result = duckdb.sql("SELECT * FROM 'users_500k.parquet' WHERE age > 50").df()

# Display the first few rows to verify
print(result.head())
print(f"Total rows found by DuckDB: {len(result)}")

   user_id     city      score  active                signup_date  age  \
0        1     Lima  43.689490   False 2023-04-16 16:00:19.927578   61   
1        2   Prague  97.403462    True 2023-12-16 16:00:19.927578   78   
2        4  Nairobi  21.993544   False 2023-12-27 16:00:19.927578   75   
3        5     Lima  91.719953    True 2026-02-15 16:00:19.927578   65   
4        7    Seoul  41.789961   False 2024-06-03 16:00:19.927578   65   

   sessions     revenue  
0       401  379.897571  
1       475  558.140786  
2       142  691.242539  
3       431  942.428408  
4       441  646.145348  
Total rows found by DuckDB: 237861


----------------
# **Task 4: pandas vs DuckDB — Identical Queries*

Run the same five analytical queries in both pandas and DuckDB on the dataset from Task 1. For each query, record the code and the execution time.

- **Query 1:** Count records per city.

- **Query 2:** Average score by city, ordered by average score descending.

- **Query 3:** For each city, find the percentage of active users whose score is above 75.

- **Query 4:** Find the top 10 users by score for each city (window function / rank).

- **Query 5:** Compute a running total of scores ordered by user_id, partitioned by city.

For DuckDB, write the query in SQL using duckdb.sql(). For pandas, use native pandas operations (groupby, transform, rank, etc.).

**Step 5: Write a markdown cell comparing:**

- Which tool was easier to express each query in?
- Which was faster?
- For which queries did the difference matter most?

**Query 1: Count records per city**

In [ ]:
# Firstly,let's do this task with Pandas

In [66]:
df_full.groupby("city").size()

city
Berlin        49970
Chiang Mai    49724
Kyoto         50059
Lima          49982
Lisbon        49885
Nairobi       50216
Perth         50059
Prague        49931
Seoul         50113
Tbilisi       50061
dtype: int64

In [67]:
%timeit df_full.groupby("city").size()

59.2 ms ± 4.58 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [69]:
# Now,let's do this with DuckDB

In [74]:
import duckdb

# DuckDB ilə SQL sorğusu
query = "SELECT city, COUNT(*) FROM 'users_500k.parquet' GROUP BY city"
city_counts_db = duckdb.sql(query).df()
print(city_counts_db)

         city  count_star()
0       Perth         50059
1  Chiang Mai         49724
2      Lisbon         49885
3        Lima         49982
4     Nairobi         50216
5      Berlin         49970
6       Kyoto         50059
7      Prague         49931
8     Tbilisi         50061
9       Seoul         50113


In [77]:
%timeit duckdb.sql(query)

1.19 ms ± 37.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [78]:
# As we see DuckDB is faster than pandas

**Query 2: Average score by city, ordered by average score descending.**

In [ ]:
# Firstly,Pandas:

In [80]:
df_full.groupby("city")["score"].mean().sort_values(ascending=False)

city
Perth         50.311448
Nairobi       50.151899
Berlin        50.135854
Lima          50.075511
Lisbon        50.063780
Kyoto         50.013372
Prague        50.009568
Chiang Mai    49.996347
Tbilisi       49.994487
Seoul         49.879502
Name: score, dtype: float64

In [81]:
%timeit df_full.groupby("city")["score"].mean().sort_values(ascending=False)

68.9 ms ± 2.04 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [82]:
# Then DuckDB

In [93]:
import duckdb

query2 = """
    SELECT city, 
           AVG(score) AS average_score 
    FROM 'users_500k.parquet' 
    GROUP BY city 
    ORDER BY average_score DESC
"""

result2 = duckdb.sql(query2).df()
print(result2)

         city  average_score
0       Perth      50.311448
1     Nairobi      50.151899
2      Berlin      50.135854
3        Lima      50.075511
4      Lisbon      50.063780
5       Kyoto      50.013372
6      Prague      50.009568
7  Chiang Mai      49.996347
8     Tbilisi      49.994487
9       Seoul      49.879502


In [95]:
%timeit result2

41.2 ns ± 1.88 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


**Query 3: For each city, find the percentage of active users whose score is above 75.**

In [96]:
# For Pandas

In [108]:
high_score_active = df_full[(df_full['active'] == True) & (df_full['score'] > 75)].groupby('city').size()

total_active = df_full[df_full['active'] == True].groupby('city').size()

percentage_high = (high_score_active / total_active) * 100
print(percentage_high)

city
Berlin        25.355897
Chiang Mai    25.054554
Kyoto         24.981993
Lima          25.064785
Lisbon        24.984980
Nairobi       25.101361
Perth         25.538499
Prague        25.431900
Seoul         24.764614
Tbilisi       24.913233
dtype: float64


In [109]:
%timeit percentage_high

40.6 ns ± 3.14 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [111]:
# DuckDB

In [123]:
import duckdb

query3 = """
    SELECT city ,COUNT(*) FROM 'users_500k.parquet'
    WHERE active = True AND score>75
    GROUP BY city
    """
result3 = duckdb.sql(query3).df()
print(result3)

         city  count_star()
0  Chiang Mai          6200
1      Lisbon          6238
2        Lima          6287
3     Nairobi          6253
4      Berlin          6323
5       Seoul          6181
6       Kyoto          6243
7       Perth          6355
8      Prague          6330
9     Tbilisi          6245


In [124]:
%timeit result3

37.2 ns ± 1.42 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


**Query 4: Find the top 10 users by score for each city (window function / rank).**

In [125]:
# For Pandas

In [134]:
df.sort_values(['city', 'score'], ascending=[True, False]).groupby('city').head(10)

,user_id,city,score,active,signup_date,age,sessions,revenue
83417,83418,Berlin,99.999831,True,2024-07-12 16:00:19.927578,34,233,366.398756
332636,332637,Berlin,99.996193,True,2024-01-19 16:00:19.927578,68,340,47.543159
184307,184308,Berlin,99.993404,True,2025-11-25 16:00:19.927578,25,210,849.064602
322631,322632,Berlin,99.993183,True,2024-12-06 16:00:19.927578,59,466,818.793654
165101,165102,Berlin,99.993058,False,2024-09-05 16:00:19.927578,19,40,767.304015
...,...,...,...,...,...,...,...,...
449823,449824,Tbilisi,99.991942,False,2024-06-16 16:00:19.927578,55,178,470.805391
337936,337937,Tbilisi,99.989324,True,2024-05-13 16:00:19.927578,19,452,650.511003
400018,400019,Tbilisi,99.987878,True,2023-12-19 16:00:19.927578,32,164,822.030000
75788,75789,Tbilisi,99.981443,False,2025-07-25 16:00:19.927578,47,111,803.861159


In [135]:
# DuckDB

In [145]:
import duckdb

query4 = """
    SELECT * FROM 
    (SELECT *, 
        ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) as rnk FROM 'users_500k.parquet') 
    WHERE rnk <= 10
"""

result4 = duckdb.sql(query4).df()
result4

,user_id,city,score,active,signup_date,age,sessions,revenue,rnk
0,2439,Lima,99.995192,True,2025-04-25 16:00:19.927578,36,23,209.604865,4
1,5790,Lisbon,99.977347,True,2025-05-21 16:00:19.927578,28,5,633.115877,7
2,7101,Prague,99.998401,False,2024-09-28 16:00:19.927578,68,185,504.491046,3
3,8136,Kyoto,99.977353,False,2024-12-13 16:00:19.927578,42,179,603.289711,10
4,12514,Chiang Mai,99.986537,True,2023-07-03 16:00:19.927578,30,344,717.807498,9
...,...,...,...,...,...,...,...,...,...
95,470889,Perth,99.981591,True,2024-12-07 16:00:19.927578,77,155,737.172837,10
96,474292,Kyoto,99.992728,True,2023-10-30 16:00:19.927578,19,89,710.608033,2
97,476523,Chiang Mai,99.986165,True,2024-08-20 16:00:19.927578,49,199,967.445542,10
98,479833,Lima,99.987232,True,2023-05-24 16:00:19.927578,43,206,679.477331,10


**Query 5: Compute a running total of scores ordered by user_id, partitioned by city.**

In [ ]:
# For pandas

In [150]:
# df_full.sort_values(['city', 'user_id']).assign(
#     running_total=df_full.groupby('city')['score'].cumsum()
# )

pandas_res5['running_total'] = df_full.sort_values(['city', 'user_id']).groupby('city')['score'].cumsum()
print(pandas_res5[['city', 'user_id', 'score', 'running_total']].head(20))

       city  user_id      score  running_total
16   Berlin       17  50.925029      50.925029
19   Berlin       20  66.114687     117.039715
38   Berlin       39  69.619799     186.659515
41   Berlin       42  35.739572     222.399086
46   Berlin       47  83.911337     306.310423
53   Berlin       54   9.907207     316.217631
56   Berlin       57  29.368527     345.586157
62   Berlin       63  84.353174     429.939331
64   Berlin       65  71.137270     501.076600
72   Berlin       73  34.452551     535.529151
100  Berlin      101   3.720223     539.249374
134  Berlin      135  21.473757     560.723131
169  Berlin      170  92.311375     653.034506
177  Berlin      178  18.640988     671.675494
178  Berlin      179  24.729882     696.405376
185  Berlin      186  70.481735     766.887111
198  Berlin      199  25.681674     792.568784
218  Berlin      219  31.931815     824.500599
223  Berlin      224  95.235129     919.735728
232  Berlin      233  50.650528     970.386256


In [160]:
%timeit pandas_res5['running_total'] 

5.49 μs ± 253 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [151]:
# For DuckDB

In [159]:
import duckdb

query5 = """
      SELECT city,user_id,score,
      SUM(score) OVER (PARTITION BY city ORDER BY user_id) as running_total
      FROM 'users_500k.parquet'
"""

result5 = duckdb.sql(query5).df()
result5

,city,user_id,score,running_total
0,Nairobi,3815,0.052587,1.853519e+04
1,Nairobi,3816,5.491855,1.854068e+04
2,Nairobi,3820,1.140448,1.854182e+04
3,Nairobi,3822,15.415056,1.855724e+04
4,Nairobi,3830,96.725392,1.865396e+04
...,...,...,...,...
499995,Tbilisi,499950,3.579564,2.502685e+06
499996,Tbilisi,499967,17.995271,2.502703e+06
499997,Tbilisi,499977,35.611077,2.502739e+06
499998,Tbilisi,499987,7.211622,2.502746e+06


In [157]:
%timeit duckdb.sql(query5).df()

295 ms ± 19.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**Step 5: Write a markdown cell comparing:**

- **Which tool was easier to express each query in?**
- **Which was faster?**
- **For which queries did the difference matter most?**

1. **Ease of Expression**

- Simple Queries (Q1, Q2): Both tools are quite intuitive. Pandas is very concise for simple grouping and sorting using methods like .groupby().size().

- Complex Queries (Q4, Q5): DuckDB (SQL) is much easier to read and write when it comes to Window Functions. Expressing a "Top 10 per city" or a "Running Total" in SQL using OVER (PARTITION BY...) is more declarative and less prone to errors than chaining multiple Pandas methods (sort_values, groupby, head, cumsum).

2. **Performance (Speed)**

Based on the %timeit results:

- DuckDB consistently outperformed Pandas across all queries.

- Pandas takes longer because it often needs to load the entire dataset into memory and perform operations in a more general-purpose way.

- DuckDB is highly optimized for analytical queries on Parquet files, as it only reads the necessary columns (columnar storage) and uses vectorized execution.

---------------------
# **Task 5: Arrow as the Bridge*

- **Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().**

- **Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.**

- **Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.**

- **Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.**

- **Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.**

- **Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?**

**Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().**

**Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.**

In [161]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Step 1: Create a DataFrame and convert it into Arrow Table
df_small = pd.DataFrame({
    'id': [1, 2, 3],
    'score': [85.5, 90.0, 78.2],
    'city': ['Baku', 'Berlin', 'Kyoto'],
    'active': [True, False, True]
})

arrow_table = pa.Table.from_pandas(df_small)

# Step 2: Check the schema 
print("Pandas Dtypes:\n", df_small.dtypes)
print("\nArrow Schema:\n", arrow_table.schema)

Pandas Dtypes:
 id          int64
score     float64
city       object
active       bool
dtype: object

Arrow Schema:
 id: int64
score: double
city: string
active: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 686


====================================

**Difference between them:**

- **Pandas uses "object" for any "str"** 
- **Arrow uses "str"**

**Arrow type is more accurate than pandas** 

In [162]:
# Step 3: Write to Parquet file və read back 
pq.write_table(arrow_table, 'bridge_test.parquet')
read_back_table = pq.read_table('bridge_test.parquet')

# Step 4: Convert to Pandas and check
df_back = read_back_table.to_pandas()
print("\nData matches original?", df_small.equals(df_back))


Data matches original? True


In [163]:
# Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. 
# Print the dtypes and compare with traditional pandas dtypes.
# Step 5: pyarrow backend ilə oxumaq
df_arrow_backed = pd.read_parquet('bridge_test.parquet', dtype_backend="pyarrow")

print("\nTraditional Pandas Dtypes:\n", df_back.dtypes)
print("\nArrow-backed Pandas Dtypes:\n", df_arrow_backed.dtypes)


Traditional Pandas Dtypes:
 id          int64
score     float64
city       object
active       bool
dtype: object

Arrow-backed Pandas Dtypes:
 id         int64[pyarrow]
score     double[pyarrow]
city      string[pyarrow]
active      bool[pyarrow]
dtype: object


In [164]:
# Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. 
# How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

Apache Arrow serves as the **universal language** for high-performance analytics. Its primary role is to provide a standardized, language-independent columnar memory format that allows different tools to share data without the overhead of serialization or copying.

**How Arrow Connects the Stack:**

1. **From Parquet (Disk) to Arrow (Memory):**
Parquet is a storage format designed for efficient disk compression and columnar access. When we read a Parquet file, Apache Arrow acts as the in-memory representation of that data. Since both use a columnar format, the transition from disk to RAM is extremely fast and efficient.

2. **From Arrow to Pandas (Analysis):**
Traditionally, Pandas used NumPy-backed blocks which were often inefficient with strings and null values. By using Arrow as a bridge (via dtype_backend="pyarrow"), Pandas can now handle data more natively. This reduces memory usage and speeds up processing because the data doesn't need to be converted into older, slower formats.

3. **From Arrow to DuckDB (SQL):**
DuckDB is designed to process data in vectors. Because Arrow stores data in a similar vectorized, columnar structure, DuckDB can execute SQL queries directly on Arrow tables. This is often called "Zero-Copy" integration, meaning the data isn't duplicated or transformed when passing it from a Parquet/Arrow source to the SQL engine.


**Conclusion:**
In a modern stack, Arrow is the glue. It eliminates the "data tax" (time spent converting formats) between storage (Parquet), processing (DuckDB), and data science (Pandas). This seamless connection is why we see such significant performance gains in modern data workflows.

#### **In simpler form:**

**Why move data between Pandas and DuckDB?**

In a modern data pipeline, we don't just use one tool; we use the best tool for the specific task. This is often referred to as using "the right tool for the job."

**1. DuckDB: The "Heavy Lifter" (Speed & Scale)**

DuckDB is built for high-performance analytical queries (OLAP). When you have millions of rows and need to perform complex aggregations, joins, or filters, DuckDB is significantly faster than Pandas. It uses a vectorized query engine that processes data in batches, making it the ideal engine for the "heavy lifting" phase of data processing. You move data to DuckDB when performance and memory efficiency are the priorities.

**2. Pandas: The "Swiss Army Knife" (Flexibility & Detail)**

Pandas is the gold standard for data manipulation and exploratory data analysis (EDA). While it might be slower on massive datasets, it offers a much richer set of functions for:

- Cleaning and transformation: Handling complex string manipulations or custom Python logic.

- Integration: Seamlessly connecting with visualization libraries (Matplotlib, Seaborn) and Machine Learning frameworks (Scikit-learn, PyTorch).

- Final Analysis: Once DuckDB has reduced 5 million rows down to a meaningful summary of 100 rows, Pandas is the perfect tool to format, pivot, and visualize that result.

**3. The Role of Apache Arrow (The Zero-Copy Bridge)**

The transition between these two tools used to be expensive in terms of time and memory. However, Apache Arrow has changed this. Because both DuckDB and Pandas (with the pyarrow backend) can "speak" Arrow, they can share the same memory space.

- When DuckDB finishes a calculation, it hands the result to Pandas via Arrow.

- There is no data copying (Zero-Copy), meaning the transfer is nearly instantaneous.

**Summary:**

We use DuckDB to crunch the big data quickly and Pandas to analyze the small, meaningful results deeply. Arrow ensures that this hand-off happens without any performance penalty.